# InsightMargin: Análise de Performance e Mix de Serviços

Este documento organiza o script fornecido em blocos lógicos, cada um com seu código e uma descrição do output esperado, para facilitar a execução e compreensão em um ambiente Jupyter Notebook.

### 1. Configuração e Conexão com o Banco de Dados

Este bloco é responsável por importar as bibliotecas necessárias, carregar as variáveis de ambiente para a conexão com o banco de dados e estabelecer a conexão.

---
⚠️ **AVISO IMPORTANTE - DADOS DE TESTE**

Os dados e outputs exibidos neste notebook são **FAKES (dados simulados para demonstração)**. 
Este projeto foi desenvolvido para fins de prototipagem e exemplo.

Para usar com dados reais:
1. Configure um banco de dados real no arquivo `.env`
2. Atualize as credenciais: `DB_SERVER`, `DB_DATABASE`, `DB_USER`, `DB_PASSWORD`
3. Execute o notebook novamente

---

In [57]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Meta da margem
META = 60.0

# Carrega variáveis do arquivo .env
load_dotenv()

# Carrega config de conexao do ambiente (.env em execucao local).
driver = os.getenv('DB_DRIVER')
server = os.getenv('DB_SERVER')
database = os.getenv('DB_NAME')
trusted = os.getenv('DB_TRUSTED_CONNECTION')
trust_cert = os.getenv('DB_TRUST_SERVER_CERT')

# Criar engine SQLAlchemy
engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}?"
    f"driver={driver}&Trusted_Connection={trusted}&TrustServerCertificate={trust_cert}"
)

Output Esperado: Nenhuma saída direta. Este bloco configura o ambiente e a conexão com o banco de dados. Variáveis de ambiente são carregadas e um objeto engine para SQLAlchemy é criado.

### 2. Extração de Dados e Visualização Inicial

Este bloco executa uma consulta SQL para extrair os dados brutos do banco de dados e carrega-os em um DataFrame do Pandas, exibindo as primeiras linhas para verificação.

In [58]:
query = """
    SELECT
        d.ID_Dados,
        d.Cliente,
        d.Data,
        d.Valores,
        d.TipoPagamento,
        d.Tipo_Atendimento,
        s.Descricao AS Servico
    FROM Dados d
    INNER JOIN Servico s ON s.IDServico = d.Servico
"""
df = pd.read_sql(query, engine)

display(df.head(3))

,ID_Dados,Cliente,Data,Valores,TipoPagamento,Tipo_Atendimento,Servico
0,5ba2d52d,ac00a9ee,2026-03-19,150.00,VC,Salão,Luzes infantil
1,7805cf60,210bdd41,2026-01-09,440.00,VC,Salão,Corte Feminino e Hidratação
2,ae4e8675,66,2025-07-31,470.00,VC,Salão,Corte Feminino e Hidratação


Output Esperado: As três primeiras linhas do DataFrame df, mostrando as colunas ID_Dados, Cliente, Data, Valores, TipoPagamento, Tipo_Atendimento e Servico.

### 3. Tratamento de Dados e Regras de Negócio

Neste bloco, são realizadas transformações nos dados, como a tipagem da coluna 'Data', a criação de chaves de tempo e a aplicação de regras de negócio para calcular taxa_pgto, repasse_bruto, taxa_valor, lucro_liquido e custo_variavel.

In [59]:
# tipagem de data
df["Data"] = pd.to_datetime(df["Data"], format="%Y-%m-%d")

# Chaves de tempo
df["year_month"] = df["Data"].dt.strftime("%Y-%m")
df["Mes"] = df["Data"].dt.month_name()

# Mapeamento de taxas por tipo de pagamento
taxas = {
    "VE": 0.02,
    "VC": 0.04,
    "DINHEIRO": 0.00,
    "PIX": 0.00
}

df["taxa_pgto"] = df["TipoPagamento"].map(taxas).fillna(0)

# Repasse bruto:
# Particular: não há repasse
# Demais atendimento: 40% do valor
df["repasse_bruto"] = np.where(
    df["Tipo_Atendimento"] == "Particular",
    0,
    df["Valores"] * 0.40
)

# Taxa financeira:
# aplicada sobre os 60% que ficam para você
df["taxa_valor"] = np.where(
    df["Tipo_Atendimento"] == "Particular",
    0,
    (df["Valores"] * 0.60) * df["taxa_pgto"]
)

# Lucro Líquido:
# Particular: 100% do valor
# Demais atendimentos: 60% menos taxa
df["lucro_liquido"] = np.where(
    df["Tipo_Atendimento"] == "Particular",
    df["Valores"],
    (df["Valores"] * 0.60) - df["taxa_valor"]
)

# Custo variável total
df["custo_variavel"] = df["repasse_bruto"] + df["taxa_valor"]

df

,ID_Dados,Cliente,Data,Valores,TipoPagamento,Tipo_Atendimento,Servico,year_month,Mes,taxa_pgto,repasse_bruto,taxa_valor,lucro_liquido,custo_variavel
0,5ba2d52d,ac00a9ee,2026-03-19,150.00,VC,Salão,Luzes infantil,2026-03,March,0.04,60.00,3.60,86.40,63.60
1,7805cf60,210bdd41,2026-01-09,440.00,VC,Salão,Corte Feminino e Hidratação,2026-01,January,0.04,176.00,10.56,253.44,186.56
2,ae4e8675,66,2025-07-31,470.00,VC,Salão,Corte Feminino e Hidratação,2025-07,July,0.04,188.00,11.28,270.72,199.28
3,80eeefa6,99,2025-10-02,810.00,VE,Salão,Corte Feminino e Luzes,2025-10,October,0.02,324.00,9.72,476.28,333.72
4,4c918682,6,2026-01-07,350.00,VC,Salão,Corte Feminino e Luzes,2026-01,January,0.04,140.00,8.40,201.60,148.40
5,4a47a2be,95,2025-10-01,700.00,VC,Salão,Corte Feminino e Luzes,2025-10,October,0.04,280.00,16.80,403.20,296.80
6,2aa828ee,c04343ce,2026-03-25,450.00,VC,Salão,Progressiva,2026-03,March,0.04,180.00,10.80,259.20,190.80
7,ed4644e3,97,2025-11-21,500.00,VE,Salão,Progressiva,2025-11,November,0.02,200.00,6.00,294.00,206.00
8,ff5019eb,56,2025-07-17,450.00,VC,Salão,Progressiva,2025-07,July,0.04,180.00,10.80,259.20,190.80
9,f88331b5,24,2026-02-26,410.00,VC,Salão,"Corte Masculino, Coloração, Hidratação",2026-02,February,0.04,164.00,9.84,236.16,173.84


Output Esperado: O DataFrame df completo, com as novas colunas year_month, Mes, taxa_pgto, repasse_bruto, taxa_valor, lucro_liquido e custo_variavel adicionadas e preenchidas de acordo com as regras de negócio. O display(df) no final do bloco mostraria o DataFrame inteiro.

### 4. Validação de Cálculos

Este bloco adiciona uma coluna de verificação (check) para validar se a soma do lucro_liquido, repasse_bruto e taxa_valor é igual ao Valores original, e então exibe a contagem de valores únicos na coluna check para identificar possíveis desvios.

In [60]:
df[[
    "Valores",
    "TipoPagamento",
    "Tipo_Atendimento",
    "taxa_pgto",
    "repasse_bruto",
    "taxa_valor",
    "lucro_liquido",
    "custo_variavel"
]].head(10)

df["check"] = round(
    df["lucro_liquido"] + df["repasse_bruto"] + df["taxa_valor"] - df["Valores"], 2
)

df["check"].value_counts()

check
0.00    308
Name: count, dtype: int64

### Output Esperado:

1. As 10 primeiras linhas das colunas Valores, TipoPagamento, Tipo_Atendimento, taxa_pgto, repasse_bruto, taxa_valor, lucro_liquido e custo_variavel.

2. A contagem de ocorrências de cada valor único na coluna check. Idealmente, a maioria (ou todos) os valores deveriam ser 0.00, indicando que os cálculos estão corretos.

### 5. Agregação Mensal e KPIs Executivos

Este bloco agrupa os dados por mês/ano (year_month) para calcular KPIs executivos como faturamento, lucro, repasse, taxa, quantidade de atendimentos e ticket médio. Em seguida, calcula métricas percentuais como pct_repasse, pct_taxa e margem, e compara a margem com a média e mediana.

In [61]:
df_table = (
    df
    .groupby("year_month", as_index=False)
    .agg(
        faturamento=("Valores", "sum"),
        lucro=("lucro_liquido", "sum"),
        repasse=("repasse_bruto", "sum"),
        taxa=("taxa_valor", "sum"),
        qtd=("ID_Dados", "count"),
        ticket=("lucro_liquido", "mean")
    )
)

df_table["pct_repasse"] = (df_table["repasse"] / df_table["faturamento"]) * 100
df_table["pct_taxa"] = (df_table["taxa"] / df_table["faturamento"]) * 100
df_table["margem"] = (df_table["lucro"] / df_table["faturamento"]) * 100

df_table["margem_vs_media"] = df_table["margem"] - df_table["margem"].mean()
df_table["margem_vs_mediana"] = df_table["margem"] - df_table["margem"].median()

df_table["status_margem"] = np.where(
    df_table["margem_vs_media"] >= 0,
    "Acima da média",
    "Abaixo da média"
)

df_table = df_table.sort_values("year_month").reset_index(drop=True)

# arredondamento final
colunas_round = [
    "faturamento", "lucro", "repasse", "taxa",
    "ticket", "pct_repasse", "pct_taxa", "margem",
    "margem_vs_media", "margem_vs_mediana"
]

df_table[colunas_round] = df_table[colunas_round].round(2)

display(df_table)

,year_month,faturamento,lucro,repasse,taxa,qtd,ticket,pct_repasse,pct_taxa,margem,margem_vs_media,margem_vs_mediana,status_margem
0,2025-05,3380.00,2022.36,1304.00,53.64,22,91.93,38.58,1.59,59.83,-0.59,0.00,Abaixo da média
1,2025-06,4827.00,3053.76,1716.00,57.24,33,92.54,35.55,1.19,63.26,2.85,3.43,Acima da média
2,2025-07,6180.00,3675.44,2404.00,100.56,26,141.36,38.90,1.63,59.47,-0.95,-0.36,Abaixo da média
3,2025-08,4830.00,2885.40,1884.00,60.60,26,110.98,39.01,1.25,59.74,-0.68,-0.09,Abaixo da média
4,2025-09,5420.00,3279.60,2048.00,92.40,28,117.13,37.79,1.70,60.51,0.09,0.68,Acima da média
5,2025-10,11900.00,7049.12,4632.00,218.88,62,113.70,38.92,1.84,59.24,-1.18,-0.60,Abaixo da média
6,2025-11,6960.00,4124.04,2724.00,111.96,31,133.03,39.14,1.61,59.25,-1.17,-0.58,Abaixo da média
7,2025-12,6240.00,3817.56,2328.00,94.44,30,127.25,37.31,1.51,61.18,0.76,1.35,Acima da média
8,2026-01,4690.00,2787.48,1828.00,74.52,21,132.74,38.98,1.59,59.43,-0.98,-0.40,Abaixo da média
9,2026-02,3470.00,2109.20,1308.00,52.80,15,140.61,37.69,1.52,60.78,0.37,0.95,Acima da média


Output Esperado: Um DataFrame df_table contendo os KPIs agregados mensalmente, incluindo faturamento, lucro, repasse, taxa, quantidade, ticket médio, percentuais de repasse e taxa, margem, e a comparação da margem com a média e mediana. As colunas numéricas serão arredondadas para duas casas decimais.

### 6. Análise Executiva: Melhor e Pior Mês em Margem

Este bloco identifica e exibe o mês com a maior e a menor margem, fornecendo um resumo executivo desses períodos.

In [62]:
# Melhor e pior mês em margem
melhor_mes = df_table.loc[df_table["margem"].idxmax()]
pior_mes = df_table.loc[df_table["margem"].idxmin()]

print("Melhor mês em margem:")
display(melhor_mes.to_frame().T)

print("Pior mês em margem:")
display(pior_mes.to_frame().T)

print(
    f"Melhor mês: {melhor_mes['year_month']} | "
    f"Margem: {melhor_mes['margem']:.2f}% | "
    f"Faturamento: R$ {melhor_mes['faturamento']:.2f}"
)

print(
    f"Pior mês: {pior_mes['year_month']} | "
    f"Margem: {pior_mes['margem']:.2f}% | "
    f"Faturamento: R$ {pior_mes['faturamento']:.2f}"
)

Melhor mês em margem:


,year_month,faturamento,lucro,repasse,taxa,qtd,ticket,pct_repasse,pct_taxa,margem,margem_vs_media,margem_vs_mediana,status_margem
1,2025-06,4827.00,3053.76,1716.00,57.24,33,92.54,35.55,1.19,63.26,2.85,3.43,Acima da média


Pior mês em margem:


,year_month,faturamento,lucro,repasse,taxa,qtd,ticket,pct_repasse,pct_taxa,margem,margem_vs_media,margem_vs_mediana,status_margem
5,2025-10,11900.00,7049.12,4632.00,218.88,62,113.70,38.92,1.84,59.24,-1.18,-0.60,Abaixo da média


Melhor mês: 2025-06 | Margem: 63.26% | Faturamento: R$ 4827.00
Pior mês: 2025-10 | Margem: 59.24% | Faturamento: R$ 11900.00


### Output Esperado:

1. Um DataFrame de uma linha mostrando os detalhes do melhor mês em margem.
2. Um DataFrame de uma linha mostrando os detalhes do pior mês em margem.
3. Duas linhas de texto formatado, resumindo o year_month, margem e faturamento para o melhor e pior mês, respectivamente.

### 7. Visualização de Margem ao Longo do Tempo

Este bloco gera um gráfico de linha interativo usando Plotly, mostrando a evolução da margem ao longo do tempo, com uma linha de meta e destaque para os meses abaixo da meta.

In [63]:
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

META = 60.0
media = df_table["margem"].mean()

# Cor por ponto: verde = acima da meta, vermelho = abaixo
marker_colors = [
    "#2CA02C" if v >= META else "#D62728"
    for v in df_table["margem"]
]
marker_sizes = [10] * len(df_table)

fig = go.Figure()

# Banda de meta (faixa visual sutil)
fig.add_hrect(
    y0=META, y1=df_table["margem"].max() + 2,
    fillcolor="#2CA02C", opacity=0.04,
    line_width=0,
)

# Série principal
fig.add_trace(go.Scatter(
    x=df_table["year_month"],
    y=df_table["margem"],
    mode="lines+markers",
    line=dict(color="#324461", width=2),
    marker=dict(
        color=marker_colors,
        size=marker_sizes,
        line=dict(color="white", width=1.5),
    ),
    showlegend=False,
    hovertemplate="%{x}<br>Margem: <b>%{y:.1f}%</b><extra></extra>",
))

# Linha de meta
fig.add_hline(
    y=META,
    line=dict(dash="dash", color="#E07B00", width=1.5),
    annotation_text="<b>Meta 60,0%</b>",
    annotation_position="right",
    annotation_font=dict(size=11, color="#E07B00"),
)

# Labels só nos pontos abaixo da meta
for i, row in df_table.iterrows():
    if row["margem"] < META:
        fig.add_annotation(
            x=row["year_month"],
            y=row["margem"],
            text=f"<b>{row['margem']:.1f}%</b>",
            showarrow=False,
            yshift=-18,
            font=dict(size=10, color="#D62728"),
        )

# Label no último mês sempre
ultimo = df_table.iloc[-1]
fig.add_annotation(
    x=ultimo["year_month"],
    y=ultimo["margem"],
    text=f"<b>{ultimo['margem']:.1f}%</b>",
    showarrow=False,
    yshift=16,
    font=dict(size=11, color="#2CA02C"),
)

fig.update_layout(
    title=dict(
        text="<b> Margem volta acima da meta em mar/26 após 4 meses abaixo do patamar de 60%</b>",
        font=dict(size=14, color="#222222"),
        x=0, xanchor="left",
    ),
    xaxis=dict(showgrid=False, tickangle=-45, tickfont=dict(size=11)),
    yaxis=dict(
        showgrid=True,
        gridcolor="#EEEEEE",
        ticksuffix="%",
        tickfont=dict(size=11),
        range=[df_table["margem"].min() - 1.5, df_table["margem"].max() + 1.5],
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=20, r=130, t=70, b=60),
    font=dict(family="Inter, Arial, sans-serif"),
    height=420,
)

fig.show(config={"displayModeBar": False})

Output Esperado: Um gráfico de linha interativo, exibindo a margem percentual ao longo dos meses. O gráfico terá uma linha de meta (60%), pontos coloridos (verde para acima da meta, vermelho para abaixo), e rótulos para os pontos abaixo da meta e para o último mês. O título do gráfico refletirá a análise da margem.

### 8. Decomposição da Diferença de Margem (Waterfall Chart)

Este bloco calcula a diferença de margem entre o melhor e o pior mês e decompõe essa diferença em fatores como repasse, taxa e um resíduo/mix, visualizando essa decomposição através de um gráfico de cascata (Waterfall Chart).

In [64]:
melhor = df_table.loc[df_table["margem"].idxmax()]
pior   = df_table.loc[df_table["margem"].idxmin()]

print(f"Melhor: {melhor['year_month']} → {melhor['margem']:.2f}%")
print(f"Pior:   {pior['year_month']}  → {pior['margem']:.2f}%")

delta_total   = pior["margem"]   - melhor["margem"]        # negativo
delta_repasse = -(pior["pct_repasse"] - melhor["pct_repasse"])  # repasse maior → margem menor
delta_taxa    = -(pior["pct_taxa"]    - melhor["pct_taxa"])
delta_residuo = delta_total - delta_repasse - delta_taxa

# Efeito ticket (proxy de mix): variação no ticket médio normalizada
delta_ticket  = pior["ticket"] - melhor["ticket"]

steps = {
    f"Margem {melhor['year_month']}": melhor["margem"],
    "Efeito Repasse":                 delta_repasse,
    "Efeito Taxa":                    delta_taxa,
    "Resíduo / Mix":                  delta_residuo,
    f"Margem {pior['year_month']}":   pior["margem"],
}

labels  = list(steps.keys())
values  = list(steps.values())

# Montar base acumulada para waterfall
base = [0] * len(labels)
acc  = melhor["margem"]
for i in range(1, len(labels) - 1):
    base[i] = acc
    acc += values[i]
base[-1] = 0  # último é absoluto (measure = "total")

measures = ["absolute"] + ["relative"] * (len(labels) - 2) + ["absolute"]

cores_relativas = [
    "#2CA02C" if v >= 0 else "#D62728"
    for v in values[1:-1]
]

measures = ["absolute"] + ["relative"] * (len(labels) - 2) + ["absolute"]

fig = go.Figure(go.Waterfall(
    orientation="v",
    measure=measures,
    x=labels,
    y=values,
    textposition="outside",
    text=[f"{v:+.2f}pp" if i not in (0, len(values)-1) else f"{v:.2f}%" for i, v in enumerate(values)],
    connector=dict(line=dict(color="#CCCCCC", width=1, dash="dot")),
    increasing=dict(marker=dict(color="#2CA02C")),
    decreasing=dict(marker=dict(color="#D62728")),
    totals=dict(marker=dict(color="#324461")),
))

fig.update_layout(
    title=dict(
        text=f"<b>Repasse explica {abs(delta_repasse/delta_total*100):.0f}% da queda de margem entre {melhor['year_month']} e {pior['year_month']}</b>",
        font=dict(size=14, color="#222222"),
        x=0, xanchor="left",
    ),
    yaxis=dict(
        ticksuffix="%",
        showgrid=True,
        gridcolor="#EEEEEE",
        range=[pior["margem"] - 1, melhor["margem"] + 1],
    ),
    xaxis=dict(showgrid=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=20, r=20, t=70, b=40),
    height=420,
)

fig.show(config={"displayModeBar": False})

print(f"{delta_repasse}")
print(f"{delta_total}")
print(round(delta_repasse / delta_total * 100))

Melhor: 2025-06 → 63.26%
Pior:   2025-10  → 59.24%


-3.3700000000000045
-4.019999999999996
84


### Output Esperado:

1. Duas linhas de texto indicando a margem do melhor e do pior mês.

2. Um gráfico de cascata (Waterfall Chart) interativo, visualizando a decomposição da diferença de margem entre o melhor e o pior mês, com o título explicando a contribuição do repasse na queda da margem.

3. Três linhas de texto com os valores de delta_repasse, delta_total e o percentual arredondado da contribuição do repasse na queda total da margem.

### 9. Análise de Mix de Serviços e Tipo de Atendimento

Este bloco analisa o mix de serviços e tipos de atendimento para o melhor e pior mês em margem, calculando a receita por serviço e tipo de atendimento, bem como seus respectivos percentuais e tickets médios. Ele também identifica serviços exclusivos de cada mês.

In [65]:
pd.set_option("display.max_rows", None)        # mostra todas as linhas
pd.set_option("display.max_columns", None)     # mostra todas as colunas
pd.set_option("display.width", None)           # não quebra linha
pd.set_option("display.max_colwidth", None)    # não corta texto

# Bloco 2 — Mix de serviços: melhor vs pior mês 
df["Data"] = pd.to_datetime(df["Data"])
df["year_month"] = df["Data"].dt.to_period("M").astype(str)

melhor_ym = melhor["year_month"]
pior_ym   = pior["year_month"]

df_meses = df[df["year_month"].isin([melhor_ym, pior_ym])].copy()

# Receita por serviço em cada mês 
mix = (
    df_meses
    .groupby(["year_month", "Servico","TipoPagamento"])["Valores"]
    .sum()
    .reset_index()
)

# % dentro do mês
mix["total_mes"] = mix.groupby("year_month")["Valores"].transform("sum")
mix["pct"]       = mix["Valores"] / mix["total_mes"] * 100

# Serviços presentes nos dois meses (para comparação limpa)
servicos_melhor = set(mix[mix["year_month"] == melhor_ym]["Servico"])
servicos_pior   = set(mix[mix["year_month"] == pior_ym]["Servico"])

print(f"Serviços exclusivos de {melhor_ym}: {servicos_melhor - servicos_pior}")
print(f"Serviços exclusivos de {pior_ym}:  {servicos_pior - servicos_melhor}")
print(f"Serviços em comum: {(servicos_melhor & servicos_pior)}")
print(f"Qtd serviços em comum: {len(servicos_melhor & servicos_pior)}")
print()
print(mix.sort_values(["year_month", "pct"], ascending=[True, False]).to_string(index=False))

# Filtra o DataFrame mix para mostrar apenas o serviço "Escova"
escovas = mix[mix["Servico"] == "Coloração"].sort_values("pct", ascending=False)

# Mostra o resultado 1960
print(escovas)

# Filtra qualquer serviço que tenha a palavra "Escova" no nome
todos_com_escova = mix[mix["Servico"].str.contains("Corte Feminino", na=False)].sort_values("pct", ascending=False)

print(todos_com_escova)

# Tipo Atendimento - Melhor mês e pior mês
tipo = (
    df_meses
    .groupby(["year_month", "Tipo_Atendimento"])
    .agg(
        qtd=("Valores", "count"),
        receita=("Valores", "sum"),
        ticket=("Valores", "mean"),
    )
    .reset_index()
)

tipo["pct_receita"] = tipo.groupby("year_month")["receita"].transform(lambda x: x / x.sum() * 100)
tipo["pct_qtd"]     = tipo.groupby("year_month")["qtd"].transform(lambda x: x / x.sum() * 100)

print(tipo.sort_values(["year_month", "pct_receita"], ascending=[True, False]).to_string(index=False))

Serviços exclusivos de 2025-06: {'Mechas'}
Serviços exclusivos de 2025-10:  {'Corte Feminino, Coloração, Hidratação e Escova', 'Escova e Babyliss', 'Corte Masculino e Progressiva', 'Lavagem', 'Corte Feminino e Luzes', 'Babyliss', 'Corte Feminino, Coloração e Escova', 'Corte Masculino e Coloração', 'Caixinha', 'Coloração e Escova', 'Corte Feminino e Escova'}
Serviços em comum: {'Hidratação', 'Corte Infantil', 'Escova', 'Coloração', 'Corte Feminino', 'Corte Masculino'}
Qtd serviços em comum: 6

year_month                                        Servico TipoPagamento  Valores  total_mes   pct
   2025-06                                         Escova            VC   800.00    4827.00 16.57
   2025-06                                Corte Masculino           PIX   597.00    4827.00 12.37
   2025-06                                      Coloração           PIX   590.00    4827.00 12.22
   2025-06                                Corte Masculino            VC   390.00    4827.00  8.08
   2025-06  

### Output Esperado:

1. Listas de serviços exclusivos do melhor e pior mês, e serviços em comum.

2. Uma tabela formatada (mix) mostrando a receita total por serviço e tipo de pagamento para o melhor e pior mês, incluindo o percentual que cada serviço representa no total do mês.

3. Uma tabela filtrada para o serviço "Coloração", mostrando sua participação percentual.

4. Uma tabela filtrada para serviços que contêm "Corte Feminino", mostrando sua participação percentual.

5. Uma tabela formatada (tipo) mostrando a quantidade, receita e ticket médio por tipo de atendimento para o melhor e pior mês, incluindo os percentuais de receita e quantidade que cada tipo de atendimento representa no total do mês.

